# NB01: Data Collection

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250093214


## Setup

Run the cell below to ensure all required packages are installed before running all other cells.

In [2]:
import json
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

## The question

Power creep is defined as "the strengthening of [a] game and its pieces over time possibly to the point where new pieces invalidate older ones" [(Magruder, 2022)](https://journals.sagepub.com/doi/full/10.1177/15554120211050812#bibr20-15554120211050812). This occurs because developers want to keep their games fresh and exciting in order to keep players coming back [(Magic the Gathering on new cards sets)](https://magic.wizards.com/en/news/card-preview/fire-it-2019-06-21). However, too much power creep can make a game unrecognizeable. One card game that is perhaps infamous for power creep is Yu-Gi-Oh. One search on Reddit will provide many posts from both dedicated Yu-Gi-Oh subreddits and other card game subreddits discussing the power creep in Yu-Gi-Oh. Some posts are as old as five years, some as recent as two months ago, showing how pervasive and long-lasting the issue of power creep is. (Re-watch RTGame Yu-Gi-Oh video about coming back to the game). 

Yu-Gi-Oh has been around for nearly 30 years, and though power creep is an issue, it has still been going strong. Another franchise that has been going arguably stronger for the same length of time is Pokemon. It is so beloved that parents that grew up with Pokemon are now sharing it with their own children [(Amenabar, 2022)](https://www.washingtonpost.com/video-games/2022/08/10/pokemon-starter-parents-kids/). Yu-Gi-Oh has not experienced the same rite-of-passage experience as Pokemon has, in part due to how complicated the game is [(short Reddit thread about the subject)](https://www.reddit.com/r/Yugioh101/comments/onn3vm/dear_parents_do_your_kids_play_yugioh_and_is_it/). If Pokemon experiences power creep in the same way, this parental bonding method might not be as effective any more. Additionally, fans could get tired of having to buy all of the new games and DLC just to get access to all of the new, strong Pokemon. However, this would only occur if Pokemon experiences a high level of power creep.

Question: Has Pokemon experienced power creep over its thirty years of existance? If so, to what extent is power creep experienced?


## Where is the data coming from to answer this question?

All data collected to answer this question comes from [PokeAPI](https://pokeapi.co/docs/v2). This is a robust API offering endpoints for many different kinds of Pokemon data, but the three I will be collecting data on are their [Generation API](https://pokeapi.co/docs/v2#games-section), their [Pokemon Species API](https://pokeapi.co/docs/v2#pokemon-species), and their [Pokemon API](https://pokeapi.co/docs/v2#pokemon).

Suppelementary data will be retrieved from either [Bulbapedia](https://bulbapedia.bulbagarden.net/wiki/Main_Page), [Pokemon Database](https://pokemondb.net/), or [Serebii.net](https://www.serebii.net/).


## Getting a list of all Pokemon

First, I am going to use PokeAPI's Generation API to collect a list of all of the Pokemon species introduced in every generation. A species, as defined by PokeAPI, "forms the basis for at least one Pokemon." For instance, take the Pokemon Wormadam. Wormadam has three forms based on its cloak: Plant Cloak, Sandy Cloak, and Trash Cloak, but Wormadam is the underlying species of all three forms.

In [3]:
#Initializing empty lists for storage
api_calls = []
mon_name = []
gen_list = []
gen_mon = {}

#Calling the Generation API
for i in range(1,10):
    with open(f'../data/raw/pokemon_list_gen_{i}.json', mode='w') as f:
        request = requests.get(f"https://pokeapi.co/api/v2/generation/{i}")
        pokemon_json = request.json()
        json.dump(pokemon_json,f,indent=4)
    with open(f'../data/raw/pokemon_list_gen_{i}.json', mode='r') as f:
        data = json.load(f)
        df = pd.json_normalize(
            data,
            record_path= 'pokemon_species',
            meta = 'name',
            record_prefix= 'pokemon_'
        )

#Storing the API call URL and the Pokemon names for use in later API calls
    for i in range(len(df['pokemon_url'])):
        api_calls.append(df['pokemon_url'][i])
        mon_name.append(df['pokemon_name'][i])

#Creating a generation list for later usage in renaming columns
    gen_list.append(df['name'][0])

#Creating a dictionary of every Pokemon added in each Generation for later usage in assigning Generation to a dataframe
    for name in df['pokemon_name']:
        if df['name'][0] not in gen_mon.keys():
            gen_mon[df['name'][0]] = [name]
        else:
            gen_mon[df['name'][0]].append(name)


Now that I have a list of all the different Pokemon species' APIs, I am going to call each species' API to get the API calls for each Pokemon form individually (ex: all three types of Wormadam cloaks rather than just the Wormadam species as a whole). But, first, I am going to create a function to perform the remaining API calls for me.

### Pokemon Species Data

In [4]:
def call_api(call_list,name_list,file_suffix):
##Takes in a list of APIs to call, a list of names for the files, and a suffix for the file to call and store .json data from an API##
    for i in range(len(call_list)):
        with open(f'../data/raw/{name_list[i]}_{file_suffix}.json', mode='w') as f:
            request = requests.get(call_list[i])
            pokemon_json = request.json()
            json.dump(pokemon_json,f,indent=4)

In [10]:
call_api(api_calls,mon_name,'spec')

In [11]:
#Initializing empty lists for storage
stat_calls = []
form_name = []

#Getting the list of each Pokemon form from API alongside calls
for i in range(len(api_calls)):
        with open(f'../data/raw/{mon_name[i]}_spec.json', mode='r') as f:
                data = json.load(f)
                df = pd.json_normalize(
                data,
                record_path = 'varieties',
                sep = '_'
                )

        for i in range(len(df['pokemon_url'])):
                stat_calls.append(df['pokemon_url'][i])
                form_name.append(df['pokemon_name'][i])


### Pokemon Stats Data

In [12]:
call_api(stat_calls,form_name,'stat')

All of the necessary calls have been made, and all the data stored in the 'raw' folder under 'data'.